# Prognozowanie wyników meczów piłkarskich

Notebook wykorzystuje dane z serwisu [football-data.co.uk](https://www.football-data.co.uk).

**Cel:** pobranie wyników z 6 lig europejskich (5 sezonów), wytrenowanie modeli prognozujących wynik meczu (H/D/A) i wygenerowanie prognoz na najbliższą kolejkę.

**Ligi:** Premier League, La Liga, Bundesliga, Serie A, Ligue 1, Eredivisie

**Dwa modele prognoz:**
| Model | Dane z football-data.co.uk |
|-------|---------------------------|
| **Dixon-Coles** | bramki, drużyny, data, liga |
| **XGBoost** | powyższe + kursy (`AvgH/D/A`) + forma strzałów (SOT, rogi) + Elo |

**Cykl pracy:**
```
pobierz dane -> refresh (kursy+statystyki) -> trenuj -> porównaj -> prognozuj -> update po kolejce
```

## 1. Przygotowanie środowiska

Uruchom ten notebook z folderu `football_predictor` albo ustaw poniżej właściwą ścieżkę.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Zrodlo danych: football-data.co.uk (CSV)
os.environ["FOOTBALL_DATA_SOURCE"] = "csv"

PROJECT_DIR = Path(".").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config import DATA_SOURCE, LEAGUES, PREDICTIONS_PATH, DB_PATH
from data_loader import download_historical_data, download_fixtures
from database import upsert_matches, load_played_matches, load_all_matches
from features import build_training_features, FEATURE_COLUMNS
from model import train_model
from dixon_coles import train_dixon_coles
from compare_models import format_comparison_report
from predictor import get_next_round_fixtures, run_predictions, run_model_comparison
from updater import initialize_database, update_after_round, is_predicted_round_complete

print("Folder projektu:", PROJECT_DIR)
print("Zrodlo danych:", DATA_SOURCE)
print("Baza:", DB_PATH)
print("Ligi:", len(LEAGUES))
print("Cechy XGBoost:", len(FEATURE_COLUMNS) + 1)

## 2. Pobranie i odświeżenie danych

Dane z **[football-data.co.uk](https://www.football-data.co.uk)** — CSV z wynikami, kursami bukmacherskimi i statystykami meczów (strzały, rogi itd.).

In [ ]:
# Krok A: pierwsze uruchomienie (pominie jesli baza juz istnieje)
if not DB_PATH.exists() or load_all_matches().empty:
    print("Pobieranie danych historycznych (pierwsze uruchomienie)...")
    added = initialize_database()
    print(f"Dodano {added} rekordow.")
else:
    print(f"Baza istnieje: {len(load_all_matches())} meczow")

matches = load_played_matches()
needs_refresh = matches["avg_h"].notna().sum() < len(matches) * 0.9

# Krok B: odswiez kursy tylko gdy brakuje (oszczedza kilka minut!)
if needs_refresh:
    print("\nOdswiezanie kursow i statystyk...")
    data = download_historical_data(save_raw=True)
    upsert_matches(data)
    matches = load_played_matches()
else:
    print("\nKursy i statystyki juz w bazie - pomijam refresh.")

print(f"Kursy:     {matches['avg_h'].notna().sum()} / {len(matches)}")
print(f"SOT:       {matches['home_sot'].notna().sum()} / {len(matches)}")
matches.head()

## 3. Eksploracja danych

Sprawdźmy ile meczów mamy w każdej lidze i sezonie oraz rozkład wyników (H = wygrana gospodarzy, D = remis, A = wygrana gości).

In [ ]:
print("Liczba meczów wg ligi:")
display(matches.groupby("div").size().rename("mecze").to_frame())

print("\nPokrycie dodatkowych danych z football-data.co.uk:")
coverage = pd.DataFrame({
    "kursy (avg_h)": [matches["avg_h"].notna().sum()],
    "strzały celne (home_sot)": [matches["home_sot"].notna().sum()],
    "rogów (home_corners)": [matches["home_corners"].notna().sum()],
    "wszystkie mecze": [len(matches)],
})
display(coverage)

print("\nPrzykładowe kursy i statystyki:")
cols = ["date", "home_team", "away_team", "fthg", "ftag", "ftr",
        "avg_h", "avg_d", "avg_a", "home_sot", "away_sot"]
display(matches[cols].dropna(subset=["avg_h"]).head(5))

print("\nRozkład wyników:")
result_labels = {"H": "Wygrana gospodarzy", "D": "Remis", "A": "Wygrana gości"}
result_dist = matches["ftr"].value_counts(normalize=True).rename(index=result_labels)
display((result_dist * 100).round(1).rename("% meczów").to_frame())

## 4. Trening modeli

> **Wskazowka:** jesli uruchamiasz tez sekcje 8 (porownanie), mozesz **pominac te komorke** -
> sekcja 8 i tak trenuje oba modele od zera. Podwojny trening = 2x dluzsze czekanie.

In [ ]:
RUN_SECTION_4 = False  # ustaw True tylko jesli NIE uruchamiasz sekcji 8

if RUN_SECTION_4:
    training = build_training_features(matches)
    print(f"Obserwacje XGBoost: {len(training)}")
    xgb_model, xgb_metrics = train_model(training)
    print(f"XGBoost: {xgb_metrics['accuracy']:.1%} ({xgb_metrics['train_seconds']}s)")

    dc_model, dc_info = train_dixon_coles(matches)
    print(f"Dixon-Coles: {dc_info['leagues_fitted']} lig ({dc_info['train_seconds']}s)")
else:
    print("Sekcja 4 pominieta - trening w sekcji 8 (compare).")

## 5. Prognozy najbliższej kolejki

Pobieramy terminarz z `fixtures.csv` (z kursami) i generujemy prognozy oboma modelami.

In [ ]:
fixtures = download_fixtures()
next_round = get_next_round_fixtures(fixtures, played=matches)

print(f"Nadchodzące mecze (wszystkie ligi): {len(fixtures)}")
print(f"Mecze najbliższej kolejki: {len(next_round)}")

if next_round.empty:
    print("Brak nierozegranych meczów w fixtures.csv.")
else:
    display(
        next_round[["div", "date", "home_team", "away_team"]]
        .assign(liga=next_round["div"].map(LEAGUES))
        .sort_values(["date", "div"])
    )

In [ ]:
def _show_predictions(preds: pd.DataFrame, title: str) -> None:
    if preds.empty:
        print(f"{title}: brak prognoz.")
        return
    show = preds.copy()
    for col in ["prob_H", "prob_D", "prob_A", "confidence"]:
        show[col] = (show[col] * 100).round(1).astype(str) + "%"
    print(title)
    display(show)

# Prognozy XGBoost (odpowiednik: python main.py predict --model xgboost)
xgb_preds = run_predictions(model_name="xgboost")
_show_predictions(xgb_preds, "Prognozy XGBoost")

# Prognozy Dixon-Coles
dc_preds = run_predictions(model_name="dixon_coles")
_show_predictions(dc_preds, "Prognozy Dixon-Coles")

## 6. Aktualizacja po rozegraniu kolejki

Po zakończeniu kolejki program pobiera nowe wyniki, trenuje oba modele od nowa i generuje prognozy.

Ustaw `FORCE_UPDATE = True`, aby wymusić aktualizację bez czekania na komplet wyników.

In [ ]:
FORCE_UPDATE = False  # zmień na True, aby wymusić aktualizację

if PREDICTIONS_PATH.exists():
    saved_preds = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")
    complete = is_predicted_round_complete(saved_preds)
    print(f"Kolejka rozegrana w całości: {'TAK' if complete else 'NIE'}")
else:
    print("Brak zapisanych prognoz.")

result = update_after_round(force=FORCE_UPDATE)
print(result)

if result.get("updated"):
    display(run_predictions())

## 7. Podsumowanie

| Sekcja | Co robi | Odpowiednik w terminalu |
|--------|---------|-------------------------|
| 2 | Pobiera historię + kursy + statystyki | `python main.py init` / `refresh` |
| 4 | Trenuje XGBoost i Dixon-Coles | część `init` / `compare` |
| 5 | Prognozy najbliższej kolejki | `python main.py predict` |
| 6 | Aktualizacja po kolejce | `python main.py update` |
| 8 | Porównanie modeli | `python main.py compare` |

**Komendy terminala (PowerShell):**
```powershell
cd C:\Data_Science\08_Machine_learning_1\football_predictor
pip install -r requirements.txt
python main.py refresh
python main.py compare
python main.py predict --model xgboost
python main.py predict --model dixon_coles
python main.py predict --model auto
python main.py update
python main.py status
```

## 8. Porównanie: Dixon-Coles vs XGBoost

Porównanie na chronologicznym podziale **80/20** (ostatnie 20% meczów = test).

| Model | Źródło danych (football-data.co.uk) |
|-------|-------------------------------------|
| Dixon-Coles | `FTHG`, `FTAG`, `HomeTeam`, `AwayTeam`, `Date`, `Div` |
| XGBoost | powyższe + `AvgH/D/A` + forma `HST/AST/HC` + Elo |

Odpowiednik terminala: `python main.py compare`

In [ ]:
# Odpowiednik: python main.py compare
comparison = run_model_comparison()
print(format_comparison_report(comparison))

# Prognozy automatycznie lepszym modelem
auto_preds = run_predictions(model_name="auto")
if not auto_preds.empty:
    print(f"\nLepszy model: {comparison['winner']}")
    print(f"Prognozy zapisane w: {PREDICTIONS_PATH}")
    display(auto_preds)